# Hand Gestures for Radial Menu
**MediaPipe Hands** (21 landmarks) + **DollarPy** ($P recognizer)

Hands only • train once → save to JSON → load in C#

In [10]:
!pip install opencv-python mediapipe dollarpy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
import json
import cv2
import numpy as np
import urllib.request
from dollarpy import Recognizer, Template, Point

from mediapipe.tasks.python import BaseOptions
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision.core import image as mp_image

MODEL_DIR = "."
INDEX_TIP, THUMB_TIP = 8, 4

# Download models if needed
for fname, url in {
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
}.items():
    path = os.path.join(MODEL_DIR, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, path)

mp_drawing = vision.drawing_utils
mp_styles = vision.drawing_styles

## 1. Hand Gestures (webcam)

In [12]:
def get_hand_path_from_webcam(label, use_index=True, max_points=200):
    cap = cv2.VideoCapture(0)
    points, recording = [], False
    opts = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=os.path.join(MODEL_DIR, "hand_landmarker.task")),
        num_hands=1)
    hands = vision.HandLandmarker.create_from_options(opts)
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            mp_img = mp_image.Image(image_format=mp_image.ImageFormat.SRGB, data=rgb)
            r = hands.detect(mp_img)
            if r.hand_landmarks:
                for hl in r.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hl, vision.HandLandmarksConnections.HAND_CONNECTIONS,
                        mp_styles.get_default_hand_landmarks_style(),
                        mp_styles.get_default_hand_connections_style())
                    idx = INDEX_TIP if use_index else THUMB_TIP
                    x, y = hl[idx].x, hl[idx].y
                    if recording and len(points) < max_points:
                        points.append(Point(x, y, 1))
                    cv2.circle(frame, (int(x*frame.shape[1]), int(y*frame.shape[0])), 8, (0,255,0), -1)
            for i in range(1, len(points)):
                p1 = (int(points[i-1].x*frame.shape[1]), int(points[i-1].y*frame.shape[0]))
                p2 = (int(points[i].x*frame.shape[1]), int(points[i].y*frame.shape[0]))
                cv2.line(frame, p1, p2, (0,255,255), 2)
            cv2.putText(frame, f"SPACE: record | Q: quit | Points: {len(points)}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            cv2.imshow(label, frame)
            k = cv2.waitKey(1) & 0xFF
            if k == ord('q'): break
            if k == ord(' '): recording = not recording
    finally:
        hands.close()
    cap.release()
    cv2.destroyAllWindows()
    print(f"{label}: {len(points)} points")
    return points

## 2. Save / Load Templates (for C# Radial Menu)

In [13]:
# Templates saved below load from gesture_templates.json

## 3. Train (record gestures)

In [14]:
TEMPLATES_JSON = "gesture_templates.json"

def save_templates(templates, filepath=TEMPLATES_JSON):
    data = {"templates": []}
    for t in templates:
        pts = [[p.x, p.y, p.stroke_id or 1] for p in t]
        data["templates"].append({"name": t.name, "points": pts})
    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(templates)} templates to {filepath}")

def load_templates(filepath=TEMPLATES_JSON):
    if not os.path.exists(filepath):
        return []
    with open(filepath) as f:
        data = json.load(f)
    return [Template(d["name"], [Point(*p) for p in d["points"]]) for d in data["templates"]]

# Load existing or start fresh
templates = load_templates()
recognizer = Recognizer(templates) if templates else None
print("Templates:", [t.name for t in templates])

Templates: []


# Run each cell below to record one gesture (2–3 samples each for better accuracy)

In [15]:
# Record circle, rectangle, square (2–3 samples each)
pts = get_hand_path_from_webcam("Record: circle", use_index=True)
if len(pts) > 10:
    templates.append(Template("circle", pts))
    recognizer = Recognizer(templates)
    save_templates(templates)

Record: circle: 101 points
Saved 1 templates to gesture_templates.json


In [16]:
pts = get_hand_path_from_webcam("Record: rectangle", use_index=True)
if len(pts) > 10:
    templates.append(Template("rectangle", pts))
    recognizer = Recognizer(templates)
    save_templates(templates)

Record: rectangle: 122 points
Saved 2 templates to gesture_templates.json


In [17]:
pts = get_hand_path_from_webcam("Record: square", use_index=True)
if len(pts) > 10:
    templates.append(Template("square", pts))
    recognizer = Recognizer(templates)
    save_templates(templates)

Record: square: 135 points
Saved 3 templates to gesture_templates.json


## 5. Real-time Hands + Recognition

In [18]:
def run_realtime():
    templates = load_templates()
    rec = Recognizer(templates) if templates else None
    cap = cv2.VideoCapture(0)
    points_buf, recording, last_result = [], False, ""
    hands_opts = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=os.path.join(MODEL_DIR, "hand_landmarker.task")),
        num_hands=1)
    hands = vision.HandLandmarker.create_from_options(hands_opts)
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            mp_img = mp_image.Image(image_format=mp_image.ImageFormat.SRGB, data=rgb)
            r = hands.detect(mp_img)
            if r.hand_landmarks:
                for hl in r.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hl, vision.HandLandmarksConnections.HAND_CONNECTIONS,
                        mp_styles.get_default_hand_landmarks_style(),
                        mp_styles.get_default_hand_connections_style())
                    x, y = hl[INDEX_TIP].x, hl[INDEX_TIP].y
                    if recording and len(points_buf) < 300:
                        points_buf.append(Point(x, y, 1))
                    cv2.circle(frame, (int(x*frame.shape[1]), int(y*frame.shape[0])), 10, (0,255,0), -1)
            for i in range(1, len(points_buf)):
                p1 = (int(points_buf[i-1].x*frame.shape[1]), int(points_buf[i-1].y*frame.shape[0]))
                p2 = (int(points_buf[i].x*frame.shape[1]), int(points_buf[i].y*frame.shape[0]))
                cv2.line(frame, p1, p2, (0,255,255), 3)
            cv2.putText(frame, "SPACE: record | R: recognize | Q: quit", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            if last_result:
                cv2.putText(frame, f"{last_result}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            cv2.imshow("Hand Gestures - Radial Menu", frame)
            k = cv2.waitKey(1) & 0xFF
            if k == ord('q'): break
            if k == ord(' '): recording = not recording
            if k == ord('r') and points_buf and len(points_buf) > 10 and rec:
                name, score = rec.recognize(points_buf)
                last_result = f"{name} ({score:.2f})" if name else "?"
                points_buf = []
    finally:
        hands.close()
    cap.release()
    cv2.destroyAllWindows()

run_realtime()

## C# Radial Menu – JSON format

`gesture_templates.json` format (use with any $1/$P C# implementation):
```json
{
  "templates": [
    { "name": "circle", "points": [[x, y, stroke], ...] },
    { "name": "rectangle", "points": [[x, y, stroke], ...] }
  ]
}
```